## Load and Prepare Data



In [6]:
import pandas as pd

# Load the datasets
movies_df = pd.read_csv('/content/tmdb_5000_movies.csv')
credits_df = pd.read_csv('/content/tmdb_5000_credits.csv', engine='python', on_bad_lines='skip')

# Display column names to identify common merge key
print("Columns in movies_df:", movies_df.columns)
print("Columns in credits_df:", credits_df.columns)

# Rename 'movie_id' in credits_df to 'id' to match movies_df for merging
credits_df.rename(columns={'movie_id': 'id'}, inplace=True)

# Merge the two dataframes on the 'id' column
movies_df = pd.merge(movies_df, credits_df, on='id')

# Display the first few rows of the merged dataframe
print("\nFirst 5 rows of the merged dataframe:")
print(movies_df.head())

Columns in movies_df: Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')
Columns in credits_df: Index(['movie_id', 'title', 'cast', 'crew'], dtype='object')

First 5 rows of the merged dataframe:
      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   1999

## Process Features for Tags



In [7]:
import numpy as np

# Select relevant columns for processing
movies_processed_df = movies_df[['id', 'title_x', 'genres', 'keywords', 'cast', 'crew', 'overview']].copy()

# Handle potential missing values in the 'overview' column
movies_processed_df['overview'] = movies_processed_df['overview'].fillna('')

print("First 5 rows of the processed dataframe with selected columns and handled missing 'overview' values:")
print(movies_processed_df.head())

First 5 rows of the processed dataframe with selected columns and handled missing 'overview' values:
       id                                   title_x  \
0   19995                                    Avatar   
1     285  Pirates of the Caribbean: At World's End   
2  206647                                   Spectre   
3   49026                     The Dark Knight Rises   
4   49529                               John Carter   

                                              genres  \
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                            keywords  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...   
2  [{"id": 470, "name": "spy"}, {"id": 818, "name...   
3  [{"i

In [8]:
import numpy as np

# Select relevant columns for processing
movies_processed_df = movies_df[['id', 'title_x', 'genres', 'keywords', 'cast', 'crew', 'overview']].copy()

# Handle potential missing values in the 'overview' column
movies_processed_df['overview'] = movies_processed_df['overview'].fillna('')

print("First 5 rows of the processed dataframe with selected columns and handled missing 'overview' values:")
print(movies_processed_df.head())

First 5 rows of the processed dataframe with selected columns and handled missing 'overview' values:
       id                                   title_x  \
0   19995                                    Avatar   
1     285  Pirates of the Caribbean: At World's End   
2  206647                                   Spectre   
3   49026                     The Dark Knight Rises   
4   49529                               John Carter   

                                              genres  \
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                            keywords  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...   
2  [{"id": 470, "name": "spy"}, {"id": 818, "name...   
3  [{"i

In [9]:
import json

# Helper function to extract names from JSON-like string columns
def extract_names(json_string, limit=None):
    if isinstance(json_string, str) and json_string.strip(): # Check if it's a non-empty string
        try:
            list_of_dicts = json.loads(json_string)
            names = [d['name'] for d in list_of_dicts if 'name' in d]
            if limit:
                return ' '.join(names[:limit])
            return ' '.join(names)
        except json.JSONDecodeError:
            return '' # Return empty string for invalid JSON
    return '' # Return empty string for NaN or empty values

# Apply the helper function to 'genres', 'keywords', and 'cast'
movies_processed_df['genres'] = movies_processed_df['genres'].apply(extract_names)
movies_processed_df['keywords'] = movies_processed_df['keywords'].apply(extract_names)
movies_processed_df['cast'] = movies_processed_df['cast'].apply(lambda x: extract_names(x, limit=3))

print("First 5 rows of the processed dataframe with 'genres', 'keywords', and 'cast' transformed:")
print(movies_processed_df[['title_x', 'genres', 'keywords', 'cast']].head())

First 5 rows of the processed dataframe with 'genres', 'keywords', and 'cast' transformed:
                                    title_x  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                     genres  \
0  Action Adventure Fantasy Science Fiction   
1                  Adventure Fantasy Action   
2                    Action Adventure Crime   
3               Action Crime Drama Thriller   
4          Action Adventure Science Fiction   

                                            keywords  \
0  culture clash future space war space colony so...   
1  ocean drug abuse exotic island east india trad...   
2  spy based on novel secret agent sequel mi6 bri...   
3  dc comics crime fighter terrorist secret ident...   
4  based on novel mars medallion space travel pri...   

      

In [10]:
import json

# Helper function to extract the director's name from the 'crew' column
def extract_director(json_string):
    if isinstance(json_string, str) and json_string.strip():
        try:
            list_of_dicts = json.loads(json_string)
            for d in list_of_dicts:
                if d.get('job') == 'Director':
                    return d.get('name', '')
        except json.JSONDecodeError:
            pass # Handle invalid JSON, proceed to return empty string
    return '' # Return empty string if no director found or invalid input

# Apply the helper function to the 'crew' column
movies_processed_df['crew'] = movies_processed_df['crew'].apply(extract_director)

print("First 5 rows of the processed dataframe with 'crew' (director) transformed:")
print(movies_processed_df[['title_x', 'crew']].head())

First 5 rows of the processed dataframe with 'crew' (director) transformed:
                                    title_x               crew
0                                    Avatar      James Cameron
1  Pirates of the Caribbean: At World's End     Gore Verbinski
2                                   Spectre         Sam Mendes
3                     The Dark Knight Rises  Christopher Nolan
4                               John Carter     Andrew Stanton


In [11]:
movies_processed_df['tags'] = movies_processed_df['genres'] + ' ' + movies_processed_df['keywords'] + ' ' + movies_processed_df['cast'] + ' ' + movies_processed_df['crew'] + ' ' + movies_processed_df['overview']

print("First 5 rows of the dataframe with the new 'tags' column:")
print(movies_processed_df[['title_x', 'tags']].head())

First 5 rows of the dataframe with the new 'tags' column:
                                    title_x  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                                tags  
0  Action Adventure Fantasy Science Fiction cultu...  
1  Adventure Fantasy Action ocean drug abuse exot...  
2  Action Adventure Crime spy based on novel secr...  
3  Action Crime Drama Thriller dc comics crime fi...  
4  Action Adventure Science Fiction based on nove...  


## Clean Text Data



In [12]:
import re

# Convert 'tags' to string type to handle any non-string entries
movies_processed_df['tags'] = movies_processed_df['tags'].astype(str)

# 1. Convert all text to lowercase
movies_processed_df['tags'] = movies_processed_df['tags'].apply(lambda x: x.lower())

# 2. Remove punctuation and special characters, then remove extra spaces
movies_processed_df['tags'] = movies_processed_df['tags'].apply(lambda x: re.sub(r'[^a-zA-Z0-9]', ' ', x))
movies_processed_df['tags'] = movies_processed_df['tags'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

print("First 5 rows of the processed dataframe with cleaned 'tags' column:")
print(movies_processed_df[['title_x', 'tags']].head())

First 5 rows of the processed dataframe with cleaned 'tags' column:
                                    title_x  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                                tags  
0  action adventure fantasy science fiction cultu...  
1  adventure fantasy action ocean drug abuse exot...  
2  action adventure crime spy based on novel secr...  
3  action crime drama thriller dc comics crime fi...  
4  action adventure science fiction based on nove...  


## Vectorize Text Data



In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Instantiate TfidfVectorizer
tfidf = TfidfVectorizer()

# Apply TF-IDF vectorizer to the 'tags' column
tfidf_matrix = tfidf.fit_transform(movies_processed_df['tags'])

print("Shape of TF-IDF matrix:", tfidf_matrix.shape)

Shape of TF-IDF matrix: (4803, 28596)


## Calculate Cosine Similarity



In [14]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Shape of cosine similarity matrix:", cosine_sim.shape)

Shape of cosine similarity matrix: (4803, 4803)


## Implement Recommendation Function



In [15]:
import pandas as pd

# 1. Create a pandas Series mapping movie titles to their indices
movie_indices = pd.Series(movies_processed_df.index, index=movies_processed_df['title_x']).drop_duplicates()

# 2-8. Define the recommendation function
def recommend(movie_title):
    # Get the index of the movie that matches the title
    if movie_title not in movie_indices:
        print(f"Movie '{movie_title}' not found in the database.")
        return []

    idx = movie_indices[movie_title]

    # Get the pairwise similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 6 most similar movies (including itself)
    sim_scores = sim_scores[1:6] # Skip the first one which is the movie itself

    # Get the movie indices
    movie_indices_recommend = [i[0] for i in sim_scores]

    # Return the top 5 most similar movie titles
    return movies_processed_df['title_x'].iloc[movie_indices_recommend].tolist()

# 9. Test the function
print("Recommendations for 'Avatar':")
print(recommend('Avatar'))

print("\nRecommendations for 'The Dark Knight Rises':")
print(recommend('The Dark Knight Rises'))

Recommendations for 'Avatar':
['Aliens', 'Alien³', 'Moonraker', 'Mission to Mars', 'Alien']

Recommendations for 'The Dark Knight Rises':
['The Dark Knight', 'Batman Returns', 'Batman Begins', 'Batman', 'Batman Forever']


## Summary:

### Data Analysis Key Findings

The task successfully developed a content-based movie recommendation system following a series of data processing and machine learning steps:

*   **Data Integration**: Two datasets, `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv`, were loaded and merged into a single DataFrame on a common movie identifier, resulting in a combined dataset of 4803 movies.
*   **Feature Engineering for Recommendations**:
    *   Key textual features including 'genres', 'keywords', 'cast' (top 3 members), 'director' (extracted from 'crew'), and 'overview' were parsed from their original JSON-like formats.
    *   These extracted features were then concatenated into a single 'tags' column for each movie, forming the basis for similarity calculations.
*   **Text Data Preprocessing**: The 'tags' column was cleaned by converting all text to lowercase, removing punctuation, and standardizing spaces, preparing it for numerical conversion.
*   **TF-IDF Vectorization**: A TF-IDF Vectorizer was applied to the cleaned 'tags' column, transforming the text data into a numerical matrix of size (4803 movies, 28596 unique terms), effectively quantifying the importance of words in each movie's tags.
*   **Cosine Similarity Calculation**: Cosine similarity was computed from the TF-IDF matrix, yielding a (4803, 4803) matrix that represents the similarity score between every pair of movies.
*   **Recommendation Function**: A `recommend(movie_title)` function was implemented. When provided with a movie title, it leverages the pre-computed cosine similarity matrix to identify and return the top 5 most similar movies.
    *   For example, calling `recommend('Avatar')` produced recommendations like \['Aliens', 'Alien\u00b3', 'Moonraker', 'Mission to Mars', 'Alien'].
    *   Calling `recommend('The Dark Knight Rises')` yielded recommendations such as \['The Dark Knight', 'Batman Returns', 'Batman Begins', 'Batman', 'Batman Forever'].

### Insights or Next Steps

*   The developed system effectively provides content-based movie recommendations by identifying movies with similar textual descriptions and attributes. This approach is robust for recommending movies similar to ones a user already likes based on their features.
*   To enhance the recommendation quality, future steps could involve incorporating user-specific data (e.g., ratings, watch history) to build a hybrid recommendation system, which combines content-based filtering with collaborative filtering techniques.
